# SupeRR Selector Season 2 — Phase 3 Data Cleaning & Feature Engineering Verification

**Scope:** Inspect and verify the reproducible outputs generated by `src/03_data_cleaning.py` and `src/04_feature_engineering.py`.

**Guardrails:**
- Original CSV files in `data/raw/Data Pack/` remain untouched.
- All processed outputs reside in `data/processed/`; audit/reconciliation logs reside in `outputs/`.
- No pressure scores, batter/bowler reliability scores, or player rankings are calculated in this notebook.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "data" / "processed").is_dir() else cwd.parent
PROCESSED_DIR = ROOT / "data" / "processed"
OUT_DIR = ROOT / "outputs"

clean_del = pd.read_csv(PROCESSED_DIR / "clean_deliveries.csv", low_memory=False, keep_default_na=False, na_values=[""]);
clean_matches = pd.read_csv(PROCESSED_DIR / "clean_matches.csv", low_memory=False)
comp_master = pd.read_csv(PROCESSED_DIR / "competition_master.csv", low_memory=False)
player_master = pd.read_csv(PROCESSED_DIR / "player_master.csv", low_memory=False)
super_over = pd.read_csv(PROCESSED_DIR / "super_over_deliveries.csv", low_memory=False)
excl_del = pd.read_csv(PROCESSED_DIR / "excluded_competition_deliveries.csv", low_memory=False)

print(f"Primary regular deliveries : {len(clean_del):,} rows x {len(clean_del.columns)} cols")
print(f"Clean eligible matches     : {len(clean_matches):,}")
print(f"Super Over deliveries      : {len(super_over):,}")
print(f"Excluded comp deliveries   : {len(excl_del):,}")
print(f"Player master records      : {len(player_master):,}")

## 1. Automated Reconciliation & Cleaning Log

In [ ]:
display(pd.read_csv(OUT_DIR / "reconciliation_checks.csv"))
display(pd.read_csv(OUT_DIR / "cleaning_log.csv"))

## 2. Competition Eligibility & Excluded Records

In [ ]:
display(
    comp_master.groupby(["eligibility_category", "is_primary_eligible"], dropna=False)
    .agg(
        competitions=("comp_id", "nunique"),
        matches_meta=("matches_in_metadata", "sum"),
        matches_ball=("matches_with_ball_data", "sum"),
        regular_deliveries=("regular_deliveries", "sum"),
        super_over_deliveries=("super_over_deliveries", "sum"),
    )
    .reset_index()
)
display(pd.read_csv(OUT_DIR / "excluded_records.csv").head(15))

## 3. Pre-Delivery Match Phase & Chase Context Availability

Visualized using the **RR PERFORMANCE LAB** design palette (`dashboard/DESIGN_SYSTEM.md`).

In [ ]:
phase_counts = clean_del["match_phase"].value_counts().reindex(["Powerplay", "Middle", "Death", "Unavailable"]).fillna(0)

fig, ax = plt.subplots(figsize=(8.5, 4.2), facecolor="#24172C")
ax.set_facecolor("#392940")
palette = ["#E83E80", "#D8B45A", "#7C9EF2", "#B8A6C2"]
bars = ax.bar(phase_counts.index, phase_counts.values, color=palette, width=0.52)

ax.set_title("Primary Regular Deliveries by Pre-Delivery Match Phase (IPL + T20I, 2022-2026)", color="#F5F1F5", fontsize=12, fontweight="bold", pad=12)
ax.set_ylabel("Deliveries", color="#F5F1F5")
ax.tick_params(colors="#F5F1F5")
ax.grid(axis="y", color="#B8A6C2", alpha=0.18, linestyle="--")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["bottom", "left"]:
    ax.spines[spine].set_color("#B8A6C2")
    ax.spines[spine].set_alpha(0.35)

for bar, val in zip(bars, phase_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 1400, f"{int(val):,}", ha="center", color="#F5F1F5", fontweight="bold", fontsize=9.5)

plt.tight_layout()
plt.show()

print("2nd-innings chase context available:",
      int(clean_del["chase_context_available"].astype(str).str.lower().eq("true").sum()),
      "/", int((clean_del["innings"] == 2).sum()))

## 4. Anomaly Investigation & Player Alias Audit

In [ ]:
display(pd.read_csv(OUT_DIR / "anomaly_investigation.csv"))
display(pd.read_csv(OUT_DIR / "player_alias_report.csv"))